In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
print("Libraries loaded.")

Libraries loaded.


In [8]:
# Load the trained XGBoost model and supporting artifacts from Week 3
xgb_model = joblib.load('../models/xgboost_readmission_model.pkl')
scaler = joblib.load('../models/feature_scaler.pkl')
feature_names = joblib.load('../models/feature_names.pkl')

print(f"Model loaded: {type(xgb_model).__name__}")
print(f"Scaler loaded: {type(scaler).__name__}")
print(f"Features: {len(feature_names)}")

Model loaded: XGBClassifier
Scaler loaded: StandardScaler
Features: 41


In [10]:
# Load Week 2 feature-engineered dataset
df = pd.read_csv('../data/processed/cleaned_data_features.csv')
print(f"Loaded dataset: {df.shape}")

# Rebuild features the same way as notebook 03
feature_cols = [
    'age_midpoint', 'gender', 'race',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_diagnoses',
    'total_visits',
    'diagnosis_category', 'discharge_group',
    'change', 'diabetesMed', 'insulin'
]

X = df[feature_cols].copy()
y = df['readmitted_30d'].copy()

# One-hot encode (same as Week 3)
X = pd.get_dummies(X, drop_first=True)
X = X.astype({col: 'int' for col in X.select_dtypes('bool').columns})

# Train/test split — SAME random_state as Week 3 ensures we get identical test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

Loaded dataset: (98053, 54)
X_test shape: (19611, 41)
y_test shape: (19611,)


In [12]:
# Scale using the SAVED scaler (not refit — we want the same transformation as Week 3)
X_test_scaled = scaler.transform(X_test)

# Regenerate XGBoost predictions
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = xgb_model.predict(X_test_scaled)

print(f"Predictions generated for {len(y_proba_xgb):,} patients")
print(f"Risk score range: {y_proba_xgb.min():.3f} to {y_proba_xgb.max():.3f}")
print(f"Mean risk score: {y_proba_xgb.mean():.3f}")

Predictions generated for 19,611 patients
Risk score range: 0.006 to 0.967
Mean risk score: 0.431


In [14]:
# Pull readable patient data from the original df, indexed by test set
test_indices = X_test.index
dashboard_df = df.loc[test_indices].copy()

# Attach predictions
dashboard_df['risk_score'] = y_proba_xgb
dashboard_df['predicted_readmission'] = y_pred_xgb
dashboard_df['actual_readmission'] = y_test.values

# Quick sanity check
print(f"Dashboard dataset shape: {dashboard_df.shape}")
print(f"\nRisk score distribution:")
print(dashboard_df['risk_score'].describe().round(3))

Dashboard dataset shape: (19611, 57)

Risk score distribution:
count    19611.000
mean         0.431
std          0.162
min          0.006
25%          0.319
50%          0.432
75%          0.549
max          0.967
Name: risk_score, dtype: float64


In [16]:
# Risk tier — based on the score distribution + clinical utility
def assign_risk_tier(score):
    if score >= 0.60:
        return 'High'
    elif score >= 0.40:
        return 'Medium'
    else:
        return 'Low'

dashboard_df['risk_tier'] = dashboard_df['risk_score'].apply(assign_risk_tier)

# Recommended intervention by tier
intervention_map = {
    'High':   'Schedule 48-hour follow-up call + medication reconciliation + care coordinator review',
    'Medium': 'Standard discharge instructions + 7-day follow-up call',
    'Low':    'Standard discharge instructions'
}
dashboard_df['recommended_intervention'] = dashboard_df['risk_tier'].map(intervention_map)

# Check the distribution
print("Risk tier distribution:")
print(dashboard_df['risk_tier'].value_counts())
print()
print("As percentages:")
print((dashboard_df['risk_tier'].value_counts(normalize=True) * 100).round(1))

Risk tier distribution:
risk_tier
Low       8379
Medium    8123
High      3109
Name: count, dtype: int64

As percentages:
risk_tier
Low       42.7
Medium    41.4
High      15.9
Name: proportion, dtype: float64


In [18]:
# How well does each tier predict actual readmission?
tier_outcomes = dashboard_df.groupby('risk_tier').agg(
    patients=('actual_readmission', 'count'),
    actual_readmissions=('actual_readmission', 'sum'),
    readmission_rate=('actual_readmission', 'mean')
).reindex(['High', 'Medium', 'Low'])

tier_outcomes['readmission_rate'] = (tier_outcomes['readmission_rate'] * 100).round(2)
tier_outcomes['lift_vs_baseline'] = (tier_outcomes['readmission_rate'] / 11.29).round(2)

print("Actual readmission rate by risk tier:")
print(tier_outcomes)
print()
print(f"Overall base rate: 11.29%")

Actual readmission rate by risk tier:
           patients  actual_readmissions  readmission_rate  lift_vs_baseline
risk_tier                                                                   
High           3109                  663             21.33              1.89
Medium         8123                 1014             12.48              1.11
Low            8379                  536              6.40              0.57

Overall base rate: 11.29%


## Tier Validation — Risk Tiers Predict Actual Outcomes

| Tier | Patients | Actual Readmits | Rate | Lift vs Base |
|---|---|---|---|---|
| High | 3,109 | 663 | 21.33% | 1.89x |
| Medium | 8,123 | 1,014 | 12.48% | 1.11x |
| Low | 8,379 | 536 | 6.40% | 0.57x |

**Base rate:** 11.29%

**Interpretation:** Risk tiers genuinely differentiate. High-risk patients are readmitted at 3.3x the rate of Low-risk patients (21.3% vs 6.4%). The model is most useful as a *triage tool* — letting a hospital concentrate intervention resources on the 16% of patients flagged High while safely de-prioritizing the 43% flagged Low.

**Honest limitation:** The High tier is directionally useful but not a perfect filter — 79% of High-flagged patients will not actually be readmitted. The dashboard is a decision-support tool, not a definitive classifier.

In [21]:
# Select and rename columns for the dashboard
dashboard_columns = [
    'encounter_id', 'patient_nbr',
    'race', 'gender', 'age', 'age_midpoint',
    'time_in_hospital',
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_diagnoses',
    'total_visits',
    'diagnosis_category',
    'discharge_group',
    'change', 'diabetesMed', 'insulin',
    'risk_score', 'risk_tier', 'predicted_readmission',
    'actual_readmission',
    'recommended_intervention'
]

# Check which columns actually exist
available_cols = [c for c in dashboard_columns if c in dashboard_df.columns]
missing_cols = [c for c in dashboard_columns if c not in dashboard_df.columns]
print(f"Including {len(available_cols)} columns")
if missing_cols:
    print(f"Missing (will skip): {missing_cols}")

dashboard_export = dashboard_df[available_cols].copy()

# Round risk_score to 4 decimals for cleaner display in Tableau
dashboard_export['risk_score'] = dashboard_export['risk_score'].round(4)

# Save
import os
os.makedirs('../data/results', exist_ok=True)
output_path = '../data/results/dashboard_data.csv'
dashboard_export.to_csv(output_path, index=False)

print(f"\nExported: {output_path}")
print(f"Rows: {len(dashboard_export):,}")
print(f"Columns: {dashboard_export.shape[1]}")
print(f"\nFirst 3 rows:")
print(dashboard_export.head(3))

Including 22 columns

Exported: ../data/results/dashboard_data.csv
Rows: 19,611
Columns: 22

First 3 rows:
       encounter_id  patient_nbr       race  gender      age  age_midpoint  \
30403     102521676     24681060  Caucasian  Female  [70-80)            75   
39416     128741370     73661274  Caucasian  Female  [80-90)            85   
27295      93639420     15534216  Caucasian    Male  [70-80)            75   

       time_in_hospital  num_lab_procedures  num_procedures  num_medications  \
30403                 6                  47               3               47   
39416                 8                   2               0               11   
27295                 5                  59               0               12   

       number_diagnoses  total_visits diagnosis_category  \
30403                 4             0        Circulatory   
39416                 9             0     Other (V-code)   
27295                 5             2          Digestive   

                  